## M2 actualización — Nuevas variables
Integramos los 4 datasets nuevos: hospitales, oferta residencial, hogares unipersonales y proyección futura.

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Dataset maestro existente
df = pd.read_csv('dataset_maestro.csv',
                 dtype={'seccion_censal': str},
                 encoding='utf-8-sig')

# Tabla de municipios para cruzar nombres
df_muni = pd.read_csv('municipios_limpios.csv')

print(f"Dataset maestro: {df.shape[0]:,} filas, {df.shape[1]} columnas")
print(f"Columnas actuales: {df.columns.tolist()}")

Dataset maestro: 36,334 filas, 17 columnas
Columnas actuales: ['seccion_censal', 'CPRO', 'CMUN', 'latitud', 'longitud', 'total_ambos', 'vol_65plus', 'pct_65plus', 'vol_75plus', 'pct_75plus', 'cohorte_futura', 'mujeres_80plus', 'hombres_80plus', 'ratio_viudez', 'renta_persona_2024', 'renta_hogar_2024', 'solvente']


### Variable 1 — Tasa de crecimiento de mayores de 70 (proyección nacional)

In [6]:
df_proy = pd.read_excel(
    r'C:\Users\jcrs2\Documents\Akademia\akademia-residencias\proporcion_poblacion_mayor(proyeccion).xlsx',
    sheet_name='tabla-36666',
    header=None
)

# Los datos reales están a partir de la fila 6
# Columna 0 = año, columna 5 = proporción 70+, columna 6 = crecimiento anual
df_proy_clean = df_proy.iloc[6:, [0, 5, 6]].copy()
df_proy_clean.columns = ['anio', 'pct_70plus', 'crecimiento_anual']
df_proy_clean['anio'] = pd.to_numeric(df_proy_clean['anio'], errors='coerce')
df_proy_clean['crecimiento_anual'] = pd.to_numeric(df_proy_clean['crecimiento_anual'], errors='coerce')
df_proy_clean = df_proy_clean.dropna(subset=['anio']).sort_values('anio')

print("Proyección nacional de crecimiento de mayores de 70:")
print(df_proy_clean[df_proy_clean['anio'] >= 2024].to_string(index=False))

Proyección nacional de crecimiento de mayores de 70:
  anio pct_70plus  crecimiento_anual
2024.0  14.814285                NaN
2025.0  14.909884           0.006453
2026.0  15.046444           0.009159
2027.0  15.213488           0.011102
2028.0  15.460755           0.016253
2029.0  15.739327           0.018018
2030.0  16.046579           0.019521
2031.0   16.38555           0.021124
2032.0  16.720042           0.020414
2033.0  17.075434           0.021255
2034.0  17.461843           0.022630
2035.0  17.897462           0.024947
2036.0  18.305056           0.022774
2037.0  18.708804           0.022057
2038.0  19.127607           0.022385
2039.0  19.522977           0.020670


### Calcular tasa acumulada 2024-2029 y aplicar a cohorte futura

In [7]:
# Crecimiento acumulado 2024 → 2029
filas_2025_2029 = df_proy_clean[
    (df_proy_clean['anio'] >= 2025) & 
    (df_proy_clean['anio'] <= 2029)
]['crecimiento_anual'].dropna()

# Producto acumulado de (1 + tasa) para cada año
factor_acumulado = (1 + filas_2025_2029).prod()

print(f"Factor de crecimiento acumulado 2024→2029: {factor_acumulado:.4f}")
print(f"Equivale a un crecimiento total del {(factor_acumulado-1)*100:.1f}%")

# Aplicar a la cohorte futura
df['cohorte_futura_2029'] = (df['cohorte_futura'] * factor_acumulado).round(0)

print(f"\nCohorte futura actual (sin proyectar):  {df['cohorte_futura'].sum():,.0f}")
print(f"Cohorte futura 2029 (proyectada):       {df['cohorte_futura_2029'].sum():,.0f}")
print(f"Diferencia:                             {(df['cohorte_futura_2029']-df['cohorte_futura']).sum():,.0f} personas más")

Factor de crecimiento acumulado 2024→2029: 1.0624
Equivale a un crecimiento total del 6.2%

Cohorte futura actual (sin proyectar):  4,680,928
Cohorte futura 2029 (proyectada):       4,972,301
Diferencia:                             291,373 personas más


### Variable 2 — Accesibilidad sanitaria

In [8]:
df_hosp = pd.read_excel('CNH_2025.xlsx', sheet_name='DIRECTORIO DE HOSPITALES')

print(f"Total hospitales: {df_hosp.shape[0]}")
print(f"Tipos de centro: {df_hosp['Clase de Centro'].value_counts().head(8).to_string()}")

Total hospitales: 848
Tipos de centro: Clase de Centro
Hospitales Generales                                        549
Hospitales de media y larga estancia                        119
Hospitales de salud mental y tratamiento de toxicomanías     95
Hospitales especializados                                    51
Otros Centros con Internamiento                              34


### Cruzar hospitales con secciones censales

In [9]:
# Filtrar solo hospitales generales
hosp_generales = df_hosp[
    df_hosp['Clase de Centro'] == 'Hospitales Generales'
].copy()

# Limpiar códigos
hosp_generales['CPRO'] = pd.to_numeric(hosp_generales['Cód. Provincia'], errors='coerce')
hosp_generales['CMUN'] = pd.to_numeric(hosp_generales['Cód. Municipio'].astype(str).str[-3:], errors='coerce')

# Municipios con hospital general
municipios_con_hospital = hosp_generales[['CPRO', 'CMUN']].drop_duplicates()
municipios_con_hospital['tiene_hospital'] = 1

print(f"Hospitales generales: {len(hosp_generales)}")
print(f"Municipios con hospital general: {len(municipios_con_hospital)}")

# Cruzar con dataset maestro
df = df.merge(municipios_con_hospital, on=['CPRO', 'CMUN'], how='left')
df['tiene_hospital'] = df['tiene_hospital'].fillna(0).astype(int)

# Hospitales por provincia como accesibilidad regional
hosp_por_provincia = hosp_generales.groupby('CPRO').size().reset_index(name='hosp_en_provincia')
df = df.merge(hosp_por_provincia, on='CPRO', how='left')
df['hosp_en_provincia'] = df['hosp_en_provincia'].fillna(0).astype(int)

print(f"\nSecciones en municipio con hospital: {df['tiene_hospital'].sum():,}")
print(f"Secciones sin hospital en municipio: {(df['tiene_hospital']==0).sum():,}")
print(f"\nHospitales por provincia (media): {df['hosp_en_provincia'].mean():.1f}")

df[['seccion_censal', 'CPRO', 'CMUN', 
    'tiene_hospital', 'hosp_en_provincia']].head(8)

Hospitales generales: 549
Municipios con hospital general: 272

Secciones en municipio con hospital: 636
Secciones sin hospital en municipio: 35,698

Hospitales por provincia (media): 22.0


,seccion_censal,CPRO,CMUN,tiene_hospital,hosp_en_provincia
0,0100101001,1.0,1.0,0,3
1,0100101002,1.0,1.0,0,3
2,0100201001,1.0,2.0,0,3
3,0100201002,1.0,2.0,0,3
4,0100201003,1.0,2.0,0,3
5,0100201004,1.0,2.0,0,3
6,0100201005,1.0,2.0,0,3
7,0100201006,1.0,2.0,0,3


### Variable 3 — Oferta residencial por Comunidad Autónoma

In [11]:
# Leer oferta residencial
df_oferta = pd.read_excel(
    'Mix-oferta_residencial_existente.xlsx',
    sheet_name='SAR_plazas',
    header=None
)

# Los datos están entre filas 10 y 31
df_oferta_clean = df_oferta.iloc[10:32, [0, 2, 8]].copy()
df_oferta_clean.columns = ['ccaa_nombre', 'plazas_total', 'indice_cobertura']
df_oferta_clean['plazas_total']     = pd.to_numeric(df_oferta_clean['plazas_total'], errors='coerce')
df_oferta_clean['indice_cobertura'] = pd.to_numeric(df_oferta_clean['indice_cobertura'], errors='coerce')
df_oferta_clean = df_oferta_clean.dropna(subset=['plazas_total']).reset_index(drop=True)

print("Oferta residencial por CCAA:")
print(df_oferta_clean.to_string(index=False))

Oferta residencial por CCAA:
            ccaa_nombre  plazas_total  indice_cobertura
              Andalucía         46354          1.804100
                 Aragón         19212          4.095215
Asturias, Principado de         15212          3.264611
         Balears, Illes          5413          1.330207
               Canarias          9927          1.633758
              Cantabria          6411          3.119457
        Castilla y León         49430          6.037595
     Castilla-La Mancha         27385          5.259778
              Catalunya         63213          2.912269
   Comunitat Valenciana         28334          1.768381
            Extremadura         15479          3.702293
                Galicia         23457          2.466222
   Madrid, Comunidad de         55457          2.943155
      Murcia, Región de          6005          1.758456
   Navarra, C. Foral de          6417          2.660204
             País Vasco         19584          2.763839
            Araba/Á

### Cruzar oferta residencial con dataset maestro

In [12]:
# Añadir CODAUTO al dataset desde municipios
df_muni_ccaa = df_muni[['CPRO', 'CMUN', 'CODAUTO']].drop_duplicates()
df = df.merge(df_muni_ccaa, on=['CPRO', 'CMUN'], how='left')

print(f"CODAUTO nulos tras merge: {df['CODAUTO'].isnull().sum()}")

# Mapa de nombre CCAA a CODAUTO para cruzar con oferta
# CODAUTO del INE: 01=Andalucía, 02=Aragón, 03=Asturias...
mapa_ccaa = {
    'Andalucía': 1, 'Aragón': 2, 'Asturias, Principado de': 3,
    'Balears, Illes': 4, 'Canarias': 5, 'Cantabria': 6,
    'Castilla y León': 7, 'Castilla-La Mancha': 8, 'Catalunya': 9,
    'Comunitat Valenciana': 10, 'Extremadura': 11, 'Galicia': 12,
    'Madrid, Comunidad de': 13, 'Murcia, Región de': 14,
    'Navarra, C. Foral de': 15, 'País Vasco': 16,
    'Araba/Álava': 16, 'Bizkaia': 16, 'Gipuzkoa': 16,
    'La Rioja': 17, 'Ceuta': 18, 'Melilla': 19
}

df_oferta_clean['CODAUTO'] = df_oferta_clean['ccaa_nombre'].map(mapa_ccaa)

# Para País Vasco agrupamos los tres territorios en uno solo
df_oferta_ccaa = df_oferta_clean.groupby('CODAUTO').agg(
    plazas_total     = ('plazas_total', 'sum'),
    indice_cobertura = ('indice_cobertura', 'mean')
).reset_index()

# Cruzar con dataset maestro
df = df.merge(
    df_oferta_ccaa[['CODAUTO', 'indice_cobertura', 'plazas_total']],
    on='CODAUTO',
    how='left'
)

print(f"Secciones con índice de cobertura: {df['indice_cobertura'].notna().sum():,}")
print(f"Secciones sin índice de cobertura: {df['indice_cobertura'].isna().sum():,}")
print(f"\nÍndice de cobertura medio: {df['indice_cobertura'].mean():.2f}")
print(f"Máximo: {df['indice_cobertura'].max():.2f} (más saturado)")
print(f"Mínimo: {df['indice_cobertura'].min():.2f} (menos saturado)")

df[['seccion_censal', 'CODAUTO', 
    'indice_cobertura', 'plazas_total']].head(5)

CODAUTO nulos tras merge: 9
Secciones con índice de cobertura: 36,325
Secciones sin índice de cobertura: 9

Índice de cobertura medio: 2.99
Máximo: 6.04 (más saturado)
Mínimo: 0.94 (menos saturado)


,seccion_censal,CODAUTO,indice_cobertura,plazas_total
0,0100101001,16.0,2.773233,39168.0
1,0100101002,16.0,2.773233,39168.0
2,0100201001,16.0,2.773233,39168.0
3,0100201002,16.0,2.773233,39168.0
4,0100201003,16.0,2.773233,39168.0


### Variable 4 — Hogares unipersonales de mayores de 70

In [13]:
df_uni = pd.read_excel(
    r'C:\Users\jcrs2\Documents\Akademia\akademia-residencias\Hogares Unipersonales + 70 Proyeccion.xlsx',
    sheet_name='Proyección Mayores 70',
    header=None
)

# Fila 1 = cabecera, datos desde fila 2
# Columna 0 = provincia, columna 1 = sexo, columna 7 = año 2029
df_uni_clean = df_uni.iloc[2:, [0, 1, 7]].copy()
df_uni_clean.columns = ['provincia', 'sexo', 'hogares_uni_70_2029']
df_uni_clean['hogares_uni_70_2029'] = pd.to_numeric(
    df_uni_clean['hogares_uni_70_2029'], errors='coerce'
)

# Solo ambos sexos y rellenar provincia hacia abajo
df_uni_clean['provincia'] = df_uni_clean['provincia'].ffill()
df_uni_ambos = df_uni_clean[
    df_uni_clean['sexo'] == 'Ambos Sexos'
].dropna(subset=['hogares_uni_70_2029']).reset_index(drop=True)

print(f"Provincias con datos: {len(df_uni_ambos)}")
print(df_uni_ambos.head(8).to_string(index=False))

Provincias con datos: 52
     provincia        sexo  hogares_uni_70_2029
Total Nacional Ambos Sexos         2.165338e+06
        Málaga Ambos Sexos         6.674037e+04
          Jaén Ambos Sexos         3.490853e+04
       Almería Ambos Sexos         3.069284e+04
        Huelva Ambos Sexos         1.988899e+04
         Cádiz Ambos Sexos         4.795450e+04
       Sevilla Ambos Sexos         7.572821e+04
       Granada Ambos Sexos         4.528138e+04


### Cruzar hogares unipersonales con dataset maestro

In [14]:
# Eliminar la fila de Total Nacional
df_uni_ambos = df_uni_ambos[
    df_uni_ambos['provincia'] != 'Total Nacional'
].reset_index(drop=True)

# Mapa nombre provincia → CPRO (códigos INE)
mapa_provincia = {
    'Álava': 1, 'Albacete': 2, 'Alicante': 3, 'Almería': 4,
    'Ávila': 5, 'Badajoz': 6, 'Balears, Illes': 7, 'Barcelona': 8,
    'Burgos': 9, 'Cáceres': 10, 'Cádiz': 11, 'Castellón': 12,
    'Ciudad Real': 13, 'Córdoba': 14, 'Coruña, A': 15, 'Cuenca': 16,
    'Girona': 17, 'Granada': 18, 'Guadalajara': 19, 'Gipuzkoa': 20,
    'Huelva': 21, 'Huesca': 22, 'Jaén': 23, 'León': 24,
    'Lleida': 25, 'Rioja, La': 26, 'Lugo': 27, 'Madrid': 28,
    'Málaga': 29, 'Murcia': 30, 'Navarra': 31, 'Ourense': 32,
    'Asturias': 33, 'Palencia': 34, 'Palmas, Las': 35, 'Pontevedra': 36,
    'Salamanca': 37, 'Santa Cruz de Tenerife': 38, 'Cantabria': 39,
    'Segovia': 40, 'Sevilla': 41, 'Soria': 42, 'Tarragona': 43,
    'Teruel': 44, 'Toledo': 45, 'Valencia': 46, 'Valladolid': 47,
    'Bizkaia': 48, 'Zamora': 49, 'Zaragoza': 50, 'Ceuta': 51,
    'Melilla': 52
}

df_uni_ambos['CPRO'] = df_uni_ambos['provincia'].map(mapa_provincia)

# Ver cuántas provincias no han mapeado
sin_mapa = df_uni_ambos[df_uni_ambos['CPRO'].isna()]
print(f"Provincias sin mapear: {len(sin_mapa)}")
if len(sin_mapa) > 0:
    print(sin_mapa['provincia'].tolist())

print(f"\nProvincias mapeadas: {df_uni_ambos['CPRO'].notna().sum()}")
df_uni_ambos.head(5)

Provincias sin mapear: 4
['Valencia/València', 'Alicante/Alacant', 'Castellón/Castelló', 'Araba/Álava']

Provincias mapeadas: 47


,provincia,sexo,hogares_uni_70_2029,CPRO
0,Málaga,Ambos Sexos,66740.37051,29.0
1,Jaén,Ambos Sexos,34908.53119,23.0
2,Almería,Ambos Sexos,30692.84195,4.0
3,Huelva,Ambos Sexos,19888.99254,21.0
4,Cádiz,Ambos Sexos,47954.50420,11.0


In [15]:
# Corregir los nombres bilingües
correcciones = {
    'Valencia/València': 46,
    'Alicante/Alacant': 3,
    'Castellón/Castelló': 12,
    'Araba/Álava': 1
}

for nombre, cpro in correcciones.items():
    df_uni_ambos.loc[df_uni_ambos['provincia'] == nombre, 'CPRO'] = cpro

print(f"Provincias sin mapear tras corrección: {df_uni_ambos['CPRO'].isna().sum()}")

# Cruzar con dataset maestro
df = df.merge(
    df_uni_ambos[['CPRO', 'hogares_uni_70_2029']],
    on='CPRO',
    how='left'
)

print(f"Secciones con hogares unipersonales: {df['hogares_uni_70_2029'].notna().sum():,}")
print(f"\nEjemplo:")
df[['seccion_censal', 'CPRO', 'hogares_uni_70_2029']].head(5)

Provincias sin mapear tras corrección: 0
Secciones con hogares unipersonales: 34,299

Ejemplo:


,seccion_censal,CPRO,hogares_uni_70_2029
0,0100101001,1.0,16196.51854
1,0100101002,1.0,16196.51854
2,0100201001,1.0,16196.51854
3,0100201002,1.0,16196.51854
4,0100201003,1.0,16196.51854


### Paso final — Guardar dataset maestro actualizado

In [16]:
print("=== RESUMEN DE VARIABLES NUEVAS ===\n")
print(f"cohorte_futura_2029:  proyección de demanda a 2029 (+6.2% sobre cohorte actual)")
print(f"tiene_hospital:       1 si el municipio tiene hospital general, 0 si no")
print(f"hosp_en_provincia:    número de hospitales generales en la provincia")
print(f"indice_cobertura:     plazas residenciales por cada 100 mayores en la CCAA")
print(f"plazas_total:         total de plazas residenciales en la CCAA")
print(f"CODAUTO:              código de comunidad autónoma")
print(f"hogares_uni_70_2029:  proyección de hogares unipersonales de 70+ en 2029")

print(f"\n=== COLUMNAS TOTALES: {df.shape[1]} ===")
print(df.columns.tolist())

# Guardar
df.to_csv('dataset_maestro.csv', index=False, encoding='utf-8-sig')
print(f"\ndataset_maestro.csv actualizado → {df.shape[0]:,} filas, {df.shape[1]} columnas")

=== RESUMEN DE VARIABLES NUEVAS ===

cohorte_futura_2029:  proyección de demanda a 2029 (+6.2% sobre cohorte actual)
tiene_hospital:       1 si el municipio tiene hospital general, 0 si no
hosp_en_provincia:    número de hospitales generales en la provincia
indice_cobertura:     plazas residenciales por cada 100 mayores en la CCAA
plazas_total:         total de plazas residenciales en la CCAA
CODAUTO:              código de comunidad autónoma
hogares_uni_70_2029:  proyección de hogares unipersonales de 70+ en 2029

=== COLUMNAS TOTALES: 24 ===
['seccion_censal', 'CPRO', 'CMUN', 'latitud', 'longitud', 'total_ambos', 'vol_65plus', 'pct_65plus', 'vol_75plus', 'pct_75plus', 'cohorte_futura', 'mujeres_80plus', 'hombres_80plus', 'ratio_viudez', 'renta_persona_2024', 'renta_hogar_2024', 'solvente', 'cohorte_futura_2029', 'tiene_hospital', 'hosp_en_provincia', 'CODAUTO', 'indice_cobertura', 'plazas_total', 'hogares_uni_70_2029']

dataset_maestro.csv actualizado → 36,901 filas, 24 columnas


In [17]:
# Verificar duplicados
duplicados = df[df.duplicated(subset=['seccion_censal'], keep=False)]
print(f"Filas con seccion_censal duplicada: {len(duplicados)}")
print(f"Secciones únicas afectadas: {duplicados['seccion_censal'].nunique()}")

# Ver un ejemplo
print("\nEjemplo de duplicado:")
print(duplicados[['seccion_censal', 'CPRO', 'CMUN', 'CODAUTO']].head(6).to_string(index=False))

Filas con seccion_censal duplicada: 1134
Secciones únicas afectadas: 567

Ejemplo de duplicado:
seccion_censal  CPRO  CMUN  CODAUTO
    3900101001  39.0   1.0      6.0
    3900101001  39.0   1.0      6.0
    3900101002  39.0   1.0      6.0
    3900101002  39.0   1.0      6.0
    3900101003  39.0   1.0      6.0
    3900101003  39.0   1.0      6.0


In [18]:
# Eliminar duplicados manteniendo la primera ocurrencia
df = df.drop_duplicates(subset=['seccion_censal'], keep='first').reset_index(drop=True)

print(f"Filas tras eliminar duplicados: {df.shape[0]:,}")
print(f"Duplicados restantes: {df.duplicated(subset=['seccion_censal']).sum()}")

# Guardar
df.to_csv('dataset_maestro.csv', index=False, encoding='utf-8-sig')
print(f"\ndataset_maestro.csv guardado → {df.shape[0]:,} filas, {df.shape[1]} columnas")

Filas tras eliminar duplicados: 36,334
Duplicados restantes: 0

dataset_maestro.csv guardado → 36,334 filas, 24 columnas


### Verificación del dataset maestro actualizado

In [19]:
# 1. HOSPITALES — Madrid debería tener muchos, provincias pequeñas pocos
print("=== HOSPITALES POR PROVINCIA (top 10) ===")
hosp_check = df.groupby('CPRO')['hosp_en_provincia'].first().sort_values(ascending=False)
print(hosp_check.head(10).to_string())

# 2. TIENE_HOSPITAL — comprobamos municipios conocidos
print("\n=== TIENE_HOSPITAL — municipios conocidos ===")
check_hosp = df.groupby(['CPRO','CMUN'])[['tiene_hospital']].first()
# Madrid capital = CPRO 28, CMUN 79
# Pozuelo = CPRO 28, CMUN 115
# Municipio rural pequeño = CPRO 42 (Soria)
for cpro, cmun, nombre in [(28, 79, 'Madrid capital'), 
                            (28, 115, 'Pozuelo de Alarcon'),
                            (42, 1, 'Soria capital')]:
    try:
        val = df[(df['CPRO']==cpro) & (df['CMUN']==cmun)]['tiene_hospital'].iloc[0]
        print(f"  {nombre}: {val}")
    except:
        print(f"  {nombre}: no encontrado")

# 3. INDICE_COBERTURA — Castilla y León debería ser el más alto (~6)
print("\n=== INDICE COBERTURA POR CODAUTO ===")
cob_check = df.groupby('CODAUTO')['indice_cobertura'].first().sort_values(ascending=False)
print(cob_check.to_string())

# 4. COHORTE_FUTURA_2029 — debe ser siempre mayor que cohorte_futura
ratio = df['cohorte_futura_2029'] / df['cohorte_futura'].replace(0, np.nan)
print(f"\n=== COHORTE FUTURA 2029 ===")
print(f"Ratio 2029/actual — media: {ratio.mean():.4f} (esperado ~1.062)")
print(f"Algún valor menor que actual: {(df['cohorte_futura_2029'] < df['cohorte_futura']).sum()}")

# 5. HOGARES UNIPERSONALES — valores razonables por provincia
print(f"\n=== HOGARES UNIPERSONALES 70+ ===")
uni_check = df.groupby('CPRO')['hogares_uni_70_2029'].first().sort_values(ascending=False)
print(uni_check.head(8).to_string())

=== HOSPITALES POR PROVINCIA (top 10) ===
CPRO
8.0     61
28.0    57
29.0    25
46.0    21
7.0     20
3.0     20
41.0    19
11.0    18
38.0    16
14.0    14

=== TIENE_HOSPITAL — municipios conocidos ===
  Madrid capital: 0
  Pozuelo de Alarcon: 0
  Soria capital: 0

=== INDICE COBERTURA POR CODAUTO ===
CODAUTO
7.0     6.037595
8.0     5.259778
2.0     4.095215
17.0    4.012123
11.0    3.702293
3.0     3.264611
6.0     3.119457
13.0    2.943155
9.0     2.912269
16.0    2.773233
15.0    2.660204
12.0    2.466222
1.0     1.804100
18.0    1.801880
10.0    1.768381
14.0    1.758456
5.0     1.633758
4.0     1.330207
19.0    0.940751

=== COHORTE FUTURA 2029 ===
Ratio 2029/actual — media: 1.0614 (esperado ~1.062)
Algún valor menor que actual: 0

=== HOGARES UNIPERSONALES 70+ ===
CPRO
28.0    259312.61460
8.0     257615.88590
46.0    124511.15320
3.0      99801.16114
41.0     75728.20561
29.0     66740.37051
33.0     65676.85113
30.0     59608.38456


In [20]:
# Ver cómo está el código de municipio en el catálogo de hospitales
print("=== HOSPITALES EN MADRID — cómo vienen los códigos ===")
hosp_madrid = df_hosp[df_hosp['Cód. Provincia'] == 28].head(10)
print(hosp_madrid[['Nombre Centro', 'Cód. Municipio', 'Municipio']].to_string(index=False))

print("\n=== CÓMO EXTRAJIMOS CMUN ===")
print("Código municipio original:", hosp_madrid['Cód. Municipio'].iloc[0])
print("Tipo:", type(hosp_madrid['Cód. Municipio'].iloc[0]))

# Ver qué CMUN tiene Madrid capital en nuestro dataset
print("\n=== MADRID CAPITAL EN DATASET MAESTRO ===")
print(df[df['CPRO']==28].groupby('CMUN').size().sort_values(ascending=False).head(5))

# Ver qué CMUN tiene Madrid capital en hospitales
print("\n=== MADRID EN HOSPITALES TRAS PROCESADO ===")
hosp_generales_mad = hosp_generales[hosp_generales['CPRO']==28]
print(hosp_generales_mad[['Cód. Municipio','CPRO','CMUN','Municipio']].head(5).to_string(index=False))

=== HOSPITALES EN MADRID — cómo vienen los códigos ===
                              Nombre Centro  Cód. Municipio          Municipio
                         Hospital HM Valles          280053  Alcalá de Henares
Hospital Universitario Príncipe de Asturias          280053  Alcalá de Henares
                   Hospital Quironsalud Sur          280072           Alcorcón
  Hospital Universitario Fundacion Alcorcón          280072           Alcorcón
            Hospital Universitario del Tajo          280133           Aranjuez
         Hospital Universitario del Sureste          280148    Arganda del Rey
    Hospital Universitario HM Monteprincipe          280225 Boadilla del Monte
                      Hospital Los Madroños          280262            Brunete
                    Hospital de La Fuenfria          280382         Cercedilla
       Centro San Juan de Dios Ciempozuelos          280409       Ciempozuelos

=== CÓMO EXTRAJIMOS CMUN ===
Código municipio original: 280053
Tipo: <class

In [21]:
# El código compuesto es PPMMM donde PP=provincia y MMM=municipio (3 dígitos con cero)
# 280079 → provincia 28, municipio 079 → como int = 79 ✓
# Pero veamos qué pasa con municipios como 280053

print("Verificación de extracción de CMUN:")
ejemplos = [280053, 280079, 280072, 310977]
for cod in ejemplos:
    cpro = int(str(cod)[:2])
    cmun_3dig = str(cod)[2:]          # últimos 4 dígitos
    cmun_int  = int(str(cod)[2:])     # como entero
    print(f"  {cod} → CPRO={cpro}, CMUN_str='{cmun_3dig}', CMUN_int={cmun_int}")

# Ahora ver qué CMUN tiene Madrid capital en el dataset maestro
print("\nSecciones de Madrid capital (CPRO=28, CMUN=79):")
print(df[(df['CPRO']==28) & (df['CMUN']==79)].shape[0], "secciones")

# Ver cómo quedó el CMUN en hospitales_generales
print("\nHospitales en Madrid capital según nuestro procesado:")
print(hosp_generales[
    (hosp_generales['CPRO']==28) & 
    (hosp_generales['CMUN']==79)
][['Municipio','CPRO','CMUN']].head(5).to_string(index=False))

Verificación de extracción de CMUN:
  280053 → CPRO=28, CMUN_str='0053', CMUN_int=53
  280079 → CPRO=28, CMUN_str='0079', CMUN_int=79
  280072 → CPRO=28, CMUN_str='0072', CMUN_int=72
  310977 → CPRO=31, CMUN_str='0977', CMUN_int=977

Secciones de Madrid capital (CPRO=28, CMUN=79):
2443 secciones

Hospitales en Madrid capital según nuestro procesado:
Empty DataFrame
Columns: [Municipio, CPRO, CMUN]
Index: []


In [22]:
# Rehacer la extracción de CPRO y CMUN desde el código compuesto original
df_hosp['CPRO'] = df_hosp['Cód. Municipio'].astype(str).str[:2].astype(int)
df_hosp['CMUN'] = df_hosp['Cód. Municipio'].astype(str).str[2:].astype(int)

hosp_generales = df_hosp[
    df_hosp['Clase de Centro'] == 'Hospitales Generales'
].copy()

# Verificar Madrid capital
print("Hospitales en Madrid capital (CMUN=79):")
print(hosp_generales[
    (hosp_generales['CPRO']==28) & 
    (hosp_generales['CMUN']==79)
][['Nombre Centro','CPRO','CMUN']].to_string(index=False))

print(f"\nRango de CMUN en hospitales: min={hosp_generales['CMUN'].min()}, max={hosp_generales['CMUN'].max()}")
print(f"Rango de CMUN en dataset:    min={df['CMUN'].min()}, max={df['CMUN'].max()}")

Hospitales en Madrid capital (CMUN=79):
Empty DataFrame
Columns: [Nombre Centro, CPRO, CMUN]
Index: []

Rango de CMUN en hospitales: min=9, max=9026
Rango de CMUN en dataset:    min=1.0, max=916.0


In [23]:
# Cruzar por nombre de municipio en lugar de código
# Normalizamos los nombres para que coincidan

def normalizar_nombre(s):
    import unicodedata
    s = str(s).lower().strip()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s

hosp_generales['municipio_norm'] = hosp_generales['Municipio'].apply(normalizar_nombre)
hosp_generales['CPRO_hosp'] = hosp_generales['Cód. Provincia'].astype(int)

df_muni['municipio_norm'] = df_muni['NOMBRE'].apply(normalizar_nombre)

# Municipios con hospital — cruzar por nombre normalizado y provincia
municipios_con_hospital = hosp_generales[['CPRO_hosp','municipio_norm']].drop_duplicates()
municipios_con_hospital.columns = ['CPRO', 'municipio_norm']
municipios_con_hospital['tiene_hospital'] = 1

# Añadir municipio_norm al df_muni para cruzar
df_muni_hosp = df_muni[['CPRO','CMUN','municipio_norm']].merge(
    municipios_con_hospital, on=['CPRO','municipio_norm'], how='left'
)
df_muni_hosp['tiene_hospital'] = df_muni_hosp['tiene_hospital'].fillna(0).astype(int)

# Verificar Madrid capital
madrid = df_muni_hosp[(df_muni_hosp['CPRO']==28) & (df_muni_hosp['CMUN']==79)]
print(f"Madrid capital tiene_hospital: {madrid['tiene_hospital'].values}")

pozuelo = df_muni_hosp[(df_muni_hosp['CPRO']==28) & (df_muni_hosp['CMUN']==115)]
print(f"Pozuelo tiene_hospital: {pozuelo['tiene_hospital'].values}")

print(f"\nMunicipios con hospital: {df_muni_hosp['tiene_hospital'].sum()}")

Madrid capital tiene_hospital: [1]
Pozuelo tiene_hospital: [1]

Municipios con hospital: 270


In [24]:
# Eliminar columnas incorrectas del merge anterior
df = df.drop(columns=['tiene_hospital', 'hosp_en_provincia'], errors='ignore')

# Hospitales por provincia
hosp_por_provincia = hosp_generales.groupby('CPRO_hosp').size().reset_index()
hosp_por_provincia.columns = ['CPRO', 'hosp_en_provincia']

# Añadir tiene_hospital desde el cruce correcto
df_muni_hosp_final = df_muni_hosp[['CPRO','CMUN','tiene_hospital']].drop_duplicates()

df = df.merge(df_muni_hosp_final, on=['CPRO','CMUN'], how='left')
df['tiene_hospital'] = df['tiene_hospital'].fillna(0).astype(int)

df = df.merge(hosp_por_provincia, on='CPRO', how='left')
df['hosp_en_provincia'] = df['hosp_en_provincia'].fillna(0).astype(int)

# Verificar
print(f"Secciones con hospital en municipio: {df['tiene_hospital'].sum():,}")
print(f"Secciones sin hospital en municipio: {(df['tiene_hospital']==0).sum():,}")

# Comprobar casos conocidos
for cpro, cmun, nombre in [(28, 79, 'Madrid capital'),
                            (28, 115, 'Pozuelo'),
                            (20, 64, 'Donostia')]:
    val = df[(df['CPRO']==cpro) & (df['CMUN']==cmun)]['tiene_hospital']
    if len(val) > 0:
        print(f"  {nombre}: {val.iloc[0]}")

print(f"\nFilas totales: {df.shape[0]:,}")
print(f"Duplicados: {df.duplicated('seccion_censal').sum()}")

Secciones con hospital en municipio: 18,532
Secciones sin hospital en municipio: 17,802
  Madrid capital: 1
  Pozuelo: 1
  Donostia: 0

Filas totales: 36,334
Duplicados: 0


In [25]:
# Verificar Donostia — buscar su CPRO y CMUN real
print("Donostia en dataset maestro:")
print(df[df['CPRO']==20].groupby('CMUN').size().sort_values(ascending=False).head(5))

print("\nDonostia en municipios:")
print(df_muni[(df_muni['CPRO']==20) & 
              (df_muni['NOMBRE'].str.contains('Donos|Sebas', case=False))
             ][['CPRO','CMUN','NOMBRE']].to_string(index=False))

print("\nDonostia en hospitales (nombre normalizado):")
print(hosp_generales[hosp_generales['CPRO_hosp']==20][
    ['Nombre Centro','Municipio','municipio_norm']
].to_string(index=False))

print("\nDonostia en df_muni (nombre normalizado):")
print(df_muni[(df_muni['CPRO']==20) & 
              (df_muni['municipio_norm'].str.contains('donos|sebas', case=False))
             ][['CPRO','CMUN','NOMBRE','municipio_norm']].to_string(index=False))

Donostia en dataset maestro:
CMUN
69.0    144
45.0     42
67.0     31
30.0     23
55.0     18
dtype: int64

Donostia en municipios:
 CPRO  CMUN                 NOMBRE
   20    69 Donostia/San Sebastián

Donostia en hospitales (nombre normalizado):
                                    Nombre Centro              Municipio         municipio_norm
      Organización Sanitaria Integrada Debagoiena     Arrasate/Mondragón     arrasate/mondragon
                             Policlínica Gipuzkoa Donostia/San Sebastián donostia/san sebastian
                  Hospital Universitario Donostia Donostia/San Sebastián donostia/san sebastian
                   Hospital Bidasoa (Osi Bidasoa)            Hondarribia            hondarribia
              Hospital de Mendaro (Osi Bajo Deba)                Mendaro                mendaro
Clínica Santa Maria de La Asunción (Inviza S. A.)                 Tolosa                 tolosa
 Hospital de Zumarraga (Osi Goierri - Alto Urola)              Zumarraga        

In [26]:
# Verificar el cruce exacto
hosp_donostia = municipios_con_hospital[
    (municipios_con_hospital['CPRO']==20) & 
    (municipios_con_hospital['municipio_norm']=='donostia/san sebastian')
]
print("Donostia en municipios_con_hospital:")
print(hosp_donostia)

muni_donostia = df_muni[
    (df_muni['CPRO']==20) & 
    (df_muni['municipio_norm']=='donostia/san sebastian')
]
print("\nDonostia en df_muni:")
print(muni_donostia[['CPRO','CMUN','NOMBRE','municipio_norm']])

# Ver tipos de dato
print("\nTipo CPRO en municipios_con_hospital:", municipios_con_hospital['CPRO'].dtype)
print("Tipo CPRO en df_muni:", df_muni['CPRO'].dtype)

Donostia en municipios_con_hospital:
     CPRO          municipio_norm  tiene_hospital
102    20  donostia/san sebastian               1

Donostia en df_muni:
      CPRO  CMUN                  NOMBRE          municipio_norm
3165    20    69  Donostia/San Sebastián  donostia/san sebastian

Tipo CPRO en municipios_con_hospital: int32
Tipo CPRO en df_muni: int64


In [27]:
# Homogeneizar tipos antes del merge
municipios_con_hospital['CPRO'] = municipios_con_hospital['CPRO'].astype(int)
df_muni['CPRO'] = df_muni['CPRO'].astype(int)
df_muni['CMUN'] = df_muni['CMUN'].astype(int)

# Rehacer el cruce
df_muni_hosp_final = df_muni[['CPRO','CMUN','municipio_norm']].merge(
    municipios_con_hospital, on=['CPRO','municipio_norm'], how='left'
).drop_duplicates(subset=['CPRO','CMUN'])

df_muni_hosp_final['tiene_hospital'] = df_muni_hosp_final['tiene_hospital'].fillna(0).astype(int)

print(f"Municipios con hospital: {df_muni_hosp_final['tiene_hospital'].sum()}")

# Verificar casos conocidos
for cpro, cmun, nombre in [(28, 79, 'Madrid capital'),
                            (28, 115, 'Pozuelo'),
                            (20, 69, 'Donostia')]:
    val = df_muni_hosp_final[
        (df_muni_hosp_final['CPRO']==cpro) & 
        (df_muni_hosp_final['CMUN']==cmun)
    ]['tiene_hospital']
    print(f"  {nombre}: {val.values[0] if len(val)>0 else 'no encontrado'}")

Municipios con hospital: 270
  Madrid capital: 1
  Pozuelo: 1
  Donostia: 1


In [28]:
# Eliminar columna incorrecta y reemplazar con la correcta
df = df.drop(columns=['tiene_hospital'], errors='ignore')

df = df.merge(
    df_muni_hosp_final[['CPRO','CMUN','tiene_hospital']],
    on=['CPRO','CMUN'],
    how='left'
)
df['tiene_hospital'] = df['tiene_hospital'].fillna(0).astype(int)

# Verificación final completa
print("=== VERIFICACIÓN FINAL ===")
print(f"Filas totales:              {df.shape[0]:,}")
print(f"Duplicados:                 {df.duplicated('seccion_censal').sum()}")
print(f"Secciones con hospital:     {df['tiene_hospital'].sum():,}")
print(f"Ratio cobertura medio:      {df['indice_cobertura'].mean():.2f}")
print(f"Cohorte 2029 > actual:      {(df['cohorte_futura_2029'] >= df['cohorte_futura']).sum():,}")
print(f"Hogares unipersonales nulos:{df['hogares_uni_70_2029'].isna().sum():,}")

# Casos conocidos
print("\n=== CASOS CONOCIDOS ===")
for cpro, cmun, nombre in [(28, 79,  'Madrid capital'),
                            (28, 115, 'Pozuelo'),
                            (20, 69,  'Donostia'),
                            (9,  59,  'Burgos')]:
    fila = df[(df['CPRO']==cpro) & (df['CMUN']==cmun)]
    if len(fila) > 0:
        print(f"  {nombre}: hospital={fila['tiene_hospital'].iloc[0]}, "
              f"cobertura={fila['indice_cobertura'].iloc[0]:.2f}, "
              f"cohorte_2029={fila['cohorte_futura_2029'].iloc[0]:.0f}")

# Guardar
df.to_csv('dataset_maestro.csv', index=False, encoding='utf-8-sig')
print(f"\ndataset_maestro.csv guardado → {df.shape[0]:,} filas, {df.shape[1]} columnas")

=== VERIFICACIÓN FINAL ===
Filas totales:              36,334
Duplicados:                 0
Secciones con hospital:     18,532
Ratio cobertura medio:      2.99
Cohorte 2029 > actual:      36,334
Hogares unipersonales nulos:2,602

=== CASOS CONOCIDOS ===
  Madrid capital: hospital=1, cobertura=2.94, cohorte_2029=152
  Pozuelo: hospital=1, cobertura=2.94, cohorte_2029=265
  Donostia: hospital=1, cobertura=2.77, cohorte_2029=86
  Burgos: hospital=1, cobertura=6.04, cohorte_2029=87

dataset_maestro.csv guardado → 36,334 filas, 24 columnas


In [29]:
# Cuántas secciones hay en los 270 municipios con hospital
secciones_con_hospital = df[df['tiene_hospital']==1]['seccion_censal'].nunique()
municipios_con_hosp = df[df['tiene_hospital']==1].groupby(['CPRO','CMUN']).ngroups

print(f"Municipios con hospital en dataset: {municipios_con_hosp}")
print(f"Secciones en esos municipios:       {secciones_con_hospital:,}")
print(f"Media secciones por municipio:      {secciones_con_hospital/municipios_con_hosp:.0f}")

# Tiene sentido? Madrid tiene ~2.400 secciones
# 270 municipios × media secciones = debe dar ~18.500
print(f"\nComprobación: 270 × {secciones_con_hospital/municipios_con_hosp:.0f} = "
      f"{270 * secciones_con_hospital/municipios_con_hosp:.0f}")

# Ver los municipios con más secciones
print("\nTop 5 municipios con más secciones:")
print(df[df['tiene_hospital']==1].groupby(
    ['CPRO','CMUN']
).size().sort_values(ascending=False).head(5))

Municipios con hospital en dataset: 270
Secciones en esos municipios:       18,532
Media secciones por municipio:      69

Comprobación: 270 × 69 = 18532

Top 5 municipios con más secciones:
CPRO  CMUN 
28.0  79.0     2443
8.0   19.0     1068
46.0  250.0     590
41.0  91.0      532
50.0  297.0     491
dtype: int64


In [30]:
# 1. ÍNDICE DE COBERTURA — verificar que Castilla y León (CODAUTO=7) 
# es el más alto y Melilla el más bajo, como vimos en los datos originales
print("=== ÍNDICE COBERTURA POR CODAUTO ===")
print(df.groupby('CODAUTO')['indice_cobertura'].first().sort_values(ascending=False).to_string())

# 2. HOGARES UNIPERSONALES — verificar que Madrid (CPRO=28) 
# tiene más que Soria (CPRO=42)
print("\n=== HOGARES UNIPERSONALES — comparativa ===")
for cpro, nombre in [(28, 'Madrid'), (8, 'Barcelona'), (42, 'Soria'), (49, 'Zamora')]:
    val = df[df['CPRO']==cpro]['hogares_uni_70_2029'].iloc[0]
    print(f"  {nombre}: {val:,.0f}")

# 3. NULOS GENERALES — ninguna variable clave debe tener demasiados nulos
print("\n=== NULOS POR VARIABLE NUEVA ===")
vars_nuevas = ['cohorte_futura_2029', 'tiene_hospital', 
               'hosp_en_provincia', 'indice_cobertura', 
               'hogares_uni_70_2029']
for v in vars_nuevas:
    nulos = df[v].isna().sum()
    pct = nulos/len(df)*100
    print(f"  {v}: {nulos} nulos ({pct:.1f}%)")

=== ÍNDICE COBERTURA POR CODAUTO ===
CODAUTO
7.0     6.037595
8.0     5.259778
2.0     4.095215
17.0    4.012123
11.0    3.702293
3.0     3.264611
6.0     3.119457
13.0    2.943155
9.0     2.912269
16.0    2.773233
15.0    2.660204
12.0    2.466222
1.0     1.804100
18.0    1.801880
10.0    1.768381
14.0    1.758456
5.0     1.633758
4.0     1.330207
19.0    0.940751

=== HOGARES UNIPERSONALES — comparativa ===
  Madrid: 259,313
  Barcelona: 257,616
  Soria: 6,366
  Zamora: 12,931

=== NULOS POR VARIABLE NUEVA ===
  cohorte_futura_2029: 0 nulos (0.0%)
  tiene_hospital: 0 nulos (0.0%)
  hosp_en_provincia: 0 nulos (0.0%)
  indice_cobertura: 9 nulos (0.0%)
  hogares_uni_70_2029: 2602 nulos (7.2%)
